# Essential services access: Anna (Person 3)
**Datathon for Social Good 2026 · Bay Area Affordability Project**

Measures how far homes in each Bay Area zip code are from groceries, food banks, pharmacies, primary care, banks, check-cashing shops and transit, builds a 0–100 access score, and combines it with rent to find zip codes that are **affordable but cut off**.

**How to run:** add the datathon Drive folder to *My Drive* as a shortcut, set `DATA_ROOT` below if your path differs, then `Runtime → Run all`.

**Outputs** (saved to `datathon_outputs` in your Drive): `essentials_by_zip.csv`, `places_clean.csv`, 4 chart PNGs. See `docs/anna_essential_services.md` for column definitions, findings and data decisions.

## 1. Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/[For Students] 8th Datathon For Social Good"

## 2. Load Google Maps Places and keep open places

In [ ]:
import pandas as pd

path = DATA_ROOT + "/google maps dataset/google_maps.csv"

cols = ["place_id", "name", "category", "all_categories",
        "lat", "lon", "rating", "reviews_count",
        "open_hours", "permanently_closed"]

df = pd.read_csv(path, usecols=cols)
print(df.shape)
df.head()

In [ ]:
closed = df["permanently_closed"].astype(str).str.lower() == "true"
df = df[~closed]
print(df.shape)

## 3. Categories
394 distinct categories. The second cell searches for check-cashing-type names.

In [ ]:
counts = df["category"].value_counts()
print(len(counts), "categories")
print(counts.head(100).to_string())

In [ ]:
mask = counts.index.str.contains("check|pawn|payday|loan|cash|money", case=False)
print(counts[mask].to_string())

## 4. Sort places into 7 service buckets
Exact category names (a keyword approach matched "atm" inside "treatment", so it was dropped). Specialists and convenience stores are left out on purpose.

In [ ]:
BUCKETS = {
    "grocery":   ["Grocery store", "Supermarket", "Asian grocery store",
                  "Indian grocery store", "Produce market"],
    "food_bank": ["Food bank"],
    "pharmacy":  ["Pharmacy"],
    "primary_care": ["Doctor", "Medical clinic", "Internist", "Pediatrician",
                     "Family practice physician", "Clinic", "Community health center",
                     "Urgent care center", "Women's health clinic", "Medical Center",
                     "Hospital"],
    "bank":      ["Bank", "Credit union"],
    "check_cashing": ["Money transfer service", "Check cashing service", "Pawn shop"],
    "transit":   ["Bus stop", "Transit station", "Transport hub"],
}

cat_to_bucket = {cat: b for b, cats in BUCKETS.items() for cat in cats}
df["bucket"] = df["category"].map(cat_to_bucket)
print(df["bucket"].value_counts(dropna=False))

## 5. Hand-check: 10 random places per bucket (seed 1)

In [ ]:
for b in BUCKETS:
    sub = df[df["bucket"] == b]
    print(f"\n===== {b} ({len(sub)} places)")
    print(sub[["name", "category"]].sample(min(10, len(sub)), random_state=1).to_string())

**Hand-check notes:** reviewed 10 random places per bucket. Data decision: `check_cashing` includes "Money transfer service" (about 1,764 of 1,836 places), since these often cash checks too.

## 6. Load Zillow home locations and keep the 9 Bay Area counties
The Zillow files cover all of California (803 zip codes before filtering), so only the 9 Bay Area counties are kept (293 zip codes after). `glob.escape` is needed because the folder name contains square brackets.

In [ ]:
import glob
import numpy as np

zillow_folder = DATA_ROOT + "/zillow housing dataset"
zillow_files = sorted(glob.glob(glob.escape(zillow_folder) + "/*.csv"))
print(zillow_files)

want = ["zpid", "zipcode", "city", "county", "latitude", "longitude"]
parts = []
for f in zillow_files:
    cols = [c for c in want if c in pd.read_csv(f, nrows=0).columns]
    parts.append(pd.read_csv(f, usecols=cols, dtype={"zipcode": str}))
homes = pd.concat(parts, ignore_index=True)

homes = homes.dropna(subset=["latitude", "longitude", "zipcode"]).drop_duplicates("zpid")
homes["zipcode"] = homes["zipcode"].str.extract(r"(\d{5})")[0]
homes = homes.dropna(subset=["zipcode"])
print(homes.shape, "homes in", homes["zipcode"].nunique(), "zip codes")

In [ ]:
BAY_AREA = ["Alameda County", "Contra Costa County", "Marin County", "Napa County",
            "San Francisco County", "San Mateo County", "Santa Clara County",
            "Solano County", "Sonoma County"]

print("Before:", homes.shape)
homes = homes[homes["county"].isin(BAY_AREA)].reset_index(drop=True)
print("After:", homes.shape, "homes in", homes["zipcode"].nunique(), "zip codes")
print(homes["county"].value_counts().to_string())

## 7. Distance from every home to the nearest service, and count within 1 mile

In [ ]:
from sklearn.neighbors import BallTree

EARTH_MI = 3958.8                      # Earth's radius in miles
places = df[df["bucket"].notna()].dropna(subset=["lat", "lon"]).copy()
home_rad = np.radians(homes[["latitude", "longitude"]].to_numpy())

# Drop places outside the Bay Area: more than 3 miles from any home
home_tree = BallTree(home_rad, metric="haversine")
d, idx = home_tree.query(np.radians(places[["lat", "lon"]].to_numpy()), k=1)
places["zipcode"] = homes["zipcode"].to_numpy()[idx[:, 0]]
places = places[d[:, 0] * EARTH_MI <= 3]
print("Places kept:", len(places))

# For each bucket: distance from each home to the nearest place, and count within 1 mile
for b in BUCKETS:
    pts = places.loc[places["bucket"] == b, ["lat", "lon"]].to_numpy()
    tree = BallTree(np.radians(pts), metric="haversine")
    dist, _ = tree.query(home_rad, k=1)
    homes[f"dist_{b}_mi"] = dist[:, 0] * EARTH_MI
    homes[f"count_{b}_1mi"] = tree.query_radius(home_rad, r=1 / EARTH_MI, count_only=True)
    print(f"{b}: median distance {homes[f'dist_{b}_mi'].median():.2f} miles")

## 8. Summary by service

In [ ]:
names = {"grocery": "Grocery store", "food_bank": "Food bank", "pharmacy": "Pharmacy",
         "primary_care": "Primary care doctor", "bank": "Bank or credit union",
         "check_cashing": "Check cashing / money transfer", "transit": "Bus or transit stop"}

summary = pd.DataFrame({
    "Service": [names[b] for b in BUCKETS],
    "Places in Bay Area": [(places["bucket"] == b).sum() for b in BUCKETS],
    "Median miles to nearest": [homes[f"dist_{b}_mi"].median() for b in BUCKETS],
    "% of homes within 1 mile": [(homes[f"dist_{b}_mi"] <= 1).mean() * 100 for b in BUCKETS],
    "% of homes over 2 miles away": [(homes[f"dist_{b}_mi"] > 2).mean() * 100 for b in BUCKETS],
}).round(2).sort_values("Median miles to nearest")

summary

## 9. Summarize by zip code (median home)

In [ ]:
metric_cols = [c for c in homes.columns if c.startswith("dist_") or c.startswith("count_")]
by_zip = homes.groupby("zipcode")[metric_cols].median()
by_zip["n_homes"] = homes.groupby("zipcode").size()
by_zip["city"] = homes.groupby("zipcode")["city"].agg(lambda s: s.mode().iat[0] if s.notna().any() else None)
by_zip["county"] = homes.groupby("zipcode")["county"].agg(lambda s: s.mode().iat[0] if s.notna().any() else None)
by_zip["low_data"] = by_zip["n_homes"] < 20      # too few homes to trust
print(by_zip.shape)

In [ ]:
print("Zips with fewer than 20 homes:", by_zip["low_data"].sum())
print("Zips with 20+ homes:", (~by_zip["low_data"]).sum())

## 10. Access score (0–100)
For grocery, pharmacy, primary care, bank and transit: rank zip codes by distance (closer = higher percentile), average the 5 ranks, multiply by 100. The score is relative to other Bay Area zip codes.

In [ ]:
# Closer = better. Each zip gets ranked per service, and the ranks are averaged into 0-100.
access_parts = ["dist_grocery_mi", "dist_pharmacy_mi", "dist_primary_care_mi",
                "dist_bank_mi", "dist_transit_mi"]
ranks = pd.concat([1 - by_zip[c].rank(pct=True) for c in access_parts], axis=1)
by_zip["access_score"] = (ranks.mean(axis=1) * 100).round(1)

# Banks per check-cashing shop within 1 mile (low = people likely pay fees to cash checks)
by_zip["bank_to_checkcash_ratio"] = by_zip["count_bank_1mi"] / (by_zip["count_check_cashing_1mi"] + 1)

ok = by_zip[~by_zip["low_data"]]
show = ["city", "access_score", "dist_grocery_mi", "dist_pharmacy_mi", "dist_primary_care_mi", "n_homes"]
print("LEAST accessible zips")
print(ok.sort_values("access_score")[show].head(10).to_string())
print("\nMOST accessible zips")
print(ok.sort_values("access_score", ascending=False)[show].head(10).to_string())

## 11. Least accessible among denser zip codes (300+ homes)

In [ ]:
urban = by_zip[by_zip["n_homes"] >= 300]
print(len(urban), "denser zip codes")
show = ["city", "access_score", "dist_grocery_mi", "dist_pharmacy_mi", "dist_primary_care_mi", "n_homes"]
urban.sort_values("access_score")[show].head(15)

## 12. Add rent and find zip codes that are affordable but cut off

In [ ]:
# Median rent per zip from Zillow rental listings
rent_parts = []
for f in zillow_files:
    r = pd.read_csv(f, usecols=["zipcode", "county", "homeStatus", "price"], dtype={"zipcode": str})
    rent_parts.append(r)
rent = pd.concat(rent_parts, ignore_index=True)
rent = rent[(rent["homeStatus"] == "FOR_RENT") & rent["county"].isin(BAY_AREA)]
rent = rent[rent["price"].between(300, 30000)]            # drop impossible rents
rent["zipcode"] = rent["zipcode"].str.extract(r"(\d{5})")[0]
by_zip["median_rent"] = rent.groupby("zipcode")["price"].median()
by_zip["n_rentals"] = rent.groupby("zipcode").size()

# Affordable (rent below the Bay Area median) but poor access (bottom third)
rel = by_zip[(~by_zip["low_data"]) & (by_zip["n_rentals"] >= 10)]
cheap = rel["median_rent"] < rel["median_rent"].median()
poor_access = rel["access_score"] < rel["access_score"].quantile(1/3)
cut_off = rel[cheap & poor_access]

print(len(cut_off), "zip codes are affordable but cut off from essentials\n")
show = ["city", "median_rent", "access_score", "dist_grocery_mi", "dist_pharmacy_mi", "dist_transit_mi"]
print(cut_off.sort_values("access_score")[show].round(2).to_string())

Rent per bedroom, so studios and in-law units in wealthy areas don't look cheaper than they are:

In [ ]:
rent_b = []
for f in zillow_files:
    rent_b.append(pd.read_csv(f, usecols=["zipcode", "county", "homeStatus", "price", "bedrooms"], dtype={"zipcode": str}))
rent_b = pd.concat(rent_b, ignore_index=True)
rent_b = rent_b[(rent_b["homeStatus"] == "FOR_RENT") & rent_b["county"].isin(BAY_AREA)
                & rent_b["price"].between(300, 30000) & (rent_b["bedrooms"] >= 1)]
rent_b["zipcode"] = rent_b["zipcode"].str.extract(r"(\d{5})")[0]
rent_b["rent_per_bedroom"] = rent_b["price"] / rent_b["bedrooms"]
by_zip["rent_per_bedroom"] = rent_b.groupby("zipcode")["rent_per_bedroom"].median()

## 13. Save results

In [ ]:
import os
out_dir = "/content/drive/MyDrive/datathon_outputs"
os.makedirs(out_dir, exist_ok=True)
by_zip.reset_index().to_csv(out_dir + "/essentials_by_zip.csv", index=False)
places.to_csv(out_dir + "/places_clean.csv", index=False)
print("Saved to", out_dir)

## 14. Charts

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.color": GRID, "grid.linewidth": 0.8,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlecolor": INK,
})

ok = by_zip[~by_zip["low_data"]].copy()
names = {"grocery": "Grocery store", "food_bank": "Food bank", "pharmacy": "Pharmacy",
         "primary_care": "Primary care doctor", "bank": "Bank or credit union",
         "check_cashing": "Check cashing / money transfer", "transit": "Bus or transit stop"}

# Chart 1: how far the typical home is from each service
med = {names[b]: homes[f"dist_{b}_mi"].median() for b in names if f"dist_{b}_mi" in homes}
med = dict(sorted(med.items(), key=lambda kv: kv[1]))
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(list(med.keys()), list(med.values()), color=BLUE, height=0.6)
for bar, v in zip(bars, med.values()):
    ax.text(v + 0.02, bar.get_y() + bar.get_height() / 2, f"{v:.2f} mi", va="center", color=INK2, fontsize=10)
ax.set_title("How far the typical Bay Area home is from each essential service")
ax.set_xlabel("Median distance from a home to the nearest one (miles)")
ax.tick_params(axis="y", length=0)
plt.tight_layout(); plt.savefig("chart1_distance_by_service.png", dpi=200); plt.show()

# Chart 2: the 10 most and 10 least accessible zip codes
low = ok.nsmallest(10, "access_score")
high = ok.nlargest(10, "access_score").iloc[::-1]
both = pd.concat([high, low.iloc[::-1]])
labels = [f"{z}  {c}" for z, c in zip(both.index, both["city"])]
colors = [BLUE] * len(high) + [ORANGE] * len(low)
fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.barh(labels, both["access_score"], color=colors, height=0.65)
for bar, v in zip(bars, both["access_score"]):
    ax.text(v + 1, bar.get_y() + bar.get_height() / 2, f"{v:.0f}", va="center", color=INK2, fontsize=10)
ax.axhline(len(low) - 0.5, color=GRID, linewidth=1)
ax.set_xlim(0, 105)
ax.set_xlabel("Access score (0 = farthest from essentials, 100 = closest)")
ax.tick_params(axis="y", length=0)
ax.legend(handles=[Patch(color=BLUE, label="Most accessible"), Patch(color=ORANGE, label="Least accessible")],
          loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
ax.set_title("The 10 most and 10 least accessible zip codes", pad=28)
plt.tight_layout(); plt.savefig("chart2_most_least_accessible.png", dpi=200); plt.show()

# Chart 3: banks vs check-cashing shops nearby
fig, ax = plt.subplots(figsize=(7, 5))
ax.grid(axis="both")
ax.scatter(ok["count_bank_1mi"], ok["count_check_cashing_1mi"], s=40, color=BLUE,
           alpha=0.75, edgecolor=SURFACE, linewidth=1)
lim = max(ok["count_bank_1mi"].max(), ok["count_check_cashing_1mi"].max()) + 1
ax.plot([0, lim], [0, lim], color=MUTED, linewidth=1, linestyle="--")
ax.text(lim * 0.97, lim * 0.9, "equal numbers", color=MUTED, ha="right", fontsize=10)
worst = ok.assign(gap=ok["count_check_cashing_1mi"] - ok["count_bank_1mi"]).nlargest(3, "gap")
for (z, r), dy in zip(worst.iterrows(), [22, 6, -10]):
    ax.annotate(f"{z} {r['city']}", (r["count_bank_1mi"], r["count_check_cashing_1mi"]),
                xytext=(14, dy), textcoords="offset points", fontsize=9, color=INK,
                arrowprops=dict(arrowstyle="-", color=MUTED, linewidth=0.8))
ax.set_title("Zips above the line have more check-cashing shops than banks")
ax.set_xlabel("Banks and credit unions within 1 mile (median home)")
ax.set_ylabel("Check-cashing / money-transfer shops within 1 mile")
plt.tight_layout(); plt.savefig("chart3_banks_vs_checkcashing.png", dpi=200); plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.color": GRID, "grid.linewidth": 0.8,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlecolor": INK,
})

# Chart 4: rent vs access, the "affordable but cut off" corner
rent_col = "rent_per_bedroom" if "rent_per_bedroom" in by_zip.columns else "median_rent"
rel = by_zip[(~by_zip["low_data"]) & (by_zip["n_rentals"] >= 10)].dropna(subset=[rent_col, "access_score"])
rent_mid = rel[rent_col].median()
access_cut = rel["access_score"].quantile(1/3)
corner = (rel[rent_col] < rent_mid) & (rel["access_score"] < access_cut)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.grid(axis="both")
ax.scatter(rel.loc[~corner, rent_col], rel.loc[~corner, "access_score"], s=36, color="#c3c2b7",
           edgecolor=SURFACE, linewidth=1, label="Other zip codes")
ax.scatter(rel.loc[corner, rent_col], rel.loc[corner, "access_score"], s=46, color=ORANGE,
           edgecolor=SURFACE, linewidth=1, label=f"Affordable but cut off ({corner.sum()} zips)")
ax.axvline(rent_mid, color=MUTED, linewidth=1, linestyle="--")
ax.axhline(access_cut, color=MUTED, linewidth=1, linestyle="--")
ax.text(rent_mid, 101, "  median rent", color=MUTED, fontsize=9, va="bottom")
ax.text(rel[rent_col].max(), access_cut, "bottom third of access  ", color=MUTED, fontsize=9,
        ha="right", va="bottom")
# Label the 4 cheapest zips in the corner
lab = rel[corner].nsmallest(4, rent_col).sort_values("access_score", ascending=False)
for (z, r), dy in zip(lab.iterrows(), [14, 4, -6, -16]):
    ax.annotate(f"{z} {r['city']}", (r[rent_col], r["access_score"]), xytext=(-14, dy),
                textcoords="offset points", fontsize=9, color=INK, ha="right",
                arrowprops=dict(arrowstyle="-", color=MUTED, linewidth=0.8))
ax.set_ylim(0, 105)
ax.set_xlim(left=rel[rent_col].min() - 700)   # room for the labels
xlabel = "Median rent per bedroom ($/month)" if rent_col == "rent_per_bedroom" else "Median rent ($/month)"
ax.set_xlabel(xlabel)
ax.set_ylabel("Access score (higher = closer to essentials)")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
ax.set_title("Orange: zip codes with cheaper rent and worse access to essentials", pad=28)
plt.tight_layout(); plt.savefig("chart4_affordable_but_cut_off.png", dpi=200); plt.show()